In [1]:
import pandas as pd
import numpy as np
final_data = pd.read_csv('finaldata.csv')

In [3]:
final_data.head()

,match_id,inning,batting_team,bowling_team,over,ball,total_runs,is_wicket,cum_runs,cum_wickets,overs_completed,current_run_rate,winner,venue_canonical,target
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,1,0,1,0,0.000000,0.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0,0,1,0,0.166667,6.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,1,0,2,0,0.333333,6.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0,0,2,0,0.500000,4.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,0,0,2,0,0.666667,3.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223


In [5]:
"Royal Challengers Bengaluru" in final_data["batting_team"].values

False

In [7]:
final_data['batting_team'].unique()

array(['Kolkata Knight Riders', 'Royal Challengers Bangalore',
       'Chennai Super Kings', 'Punjab Kings', 'Rajasthan Royals',
       'Delhi Capitals', 'Mumbai Indians', 'Sunrisers Hyderabad',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Rising Pune Supergiants',
       'Gujarat Titans', 'Lucknow Super Giants'], dtype=object)

In [9]:
final_data['bowling_team'].unique()

array(['Royal Challengers Bangalore', 'Kolkata Knight Riders',
       'Punjab Kings', 'Chennai Super Kings', 'Delhi Capitals',
       'Rajasthan Royals', 'Mumbai Indians', 'Sunrisers Hyderabad',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Rising Pune Supergiants',
       'Gujarat Titans', 'Lucknow Super Giants'], dtype=object)

In [115]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib

# For team names
le_team = LabelEncoder()
final_data['batting_team_encoded'] = le_team.fit_transform(final_data['batting_team'])
final_data['bowling_team_encoded'] = le_team.transform(final_data['bowling_team'])
# Encode the 'winner' column
final_data["winner_encoded"] = le_team.fit_transform(final_data["winner"])

# Save the team encoder
joblib.dump(le_team, 'le_team.pkl')

# For venue
le_venue = LabelEncoder()
final_data['venue_canonical_encoded'] = le_venue.fit_transform(final_data['venue_canonical'])

# Save the venue encoder
joblib.dump(le_venue, 'le_venue.pkl')

# Drop original categorical columns
final_data = final_data.drop(columns=['batting_team', 'bowling_team', 'venue_canonical'])

In [117]:
final_data = final_data.apply(pd.to_numeric, errors='coerce')


In [119]:
remaining_overs = 20 - final_data['overs_completed']
final_data['required_run_rate'] = np.where(
    (final_data['inning'] == 2) & (remaining_overs > 0),
    (final_data['target'] - final_data['cum_runs']) / remaining_overs,
    0
)
final_data['required_run_rate'] = final_data['required_run_rate'].replace([np.inf, -np.inf], 0)

# Create target variable: win = 1 if batting_team equals winner, else 0.
final_data['win'] = (final_data['batting_team_encoded'] == final_data['winner_encoded']).astype(int)

# Filter to second innings only (since required run rate applies to 2nd innings)
final_data = final_data[final_data['inning'] == 2].copy()

In [121]:
final_data = final_data.astype("str")


In [123]:
print("Batting Team Encoding:\n", dict(zip(le_team.classes_, le_team.transform(le_team.classes_))))
print("\nVenue Encoding:\n", dict(zip(le_venue.classes_, le_venue.transform(le_venue.classes_))))


Batting Team Encoding:
 {'Chennai Super Kings': 0, 'Delhi Capitals': 1, 'Gujarat Titans': 2, 'Kochi Tuskers Kerala': 3, 'Kolkata Knight Riders': 4, 'Lucknow Super Giants': 5, 'Mumbai Indians': 6, 'Pune Warriors': 7, 'Punjab Kings': 8, 'Rajasthan Royals': 9, 'Rising Pune Supergiants': 10, 'Royal Challengers Bangalore': 11, 'Sunrisers Hyderabad': 12, nan: 13}

Venue Encoding:
 {'Arun Jaitley Stadium, Delhi': 0, 'Barabati Stadium': 1, 'Barsapara Cricket Stadium, Guwahati': 2, 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 3, 'Brabourne Stadium': 4, 'Buffalo Park': 5, 'De Beers Diamond Oval': 6, 'Dr DY Patil Sports Academy': 7, 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 8, 'Dubai International Cricket Stadium': 9, 'Eden Gardens': 10, 'Green Park': 11, 'Himachal Pradesh Cricket Association Stadium': 12, 'Holkar Cricket Stadium': 13, 'JSCA International Stadium Complex': 14, 'Kingsmead': 15, 'M Chinnaswamy Stadium': 16, 'MA Chidambaram Stadium, Chepauk':

In [125]:
keep_cols = ['match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
             'required_run_rate', 'target', 'batting_team_encoded', 'bowling_team_encoded', 'winner_encoded', 'venue_canonical_encoded', 'win']
final_data = final_data[keep_cols]
final_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 125741 entries, 124 to 260919
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   match_id                 125741 non-null  object
 1   inning                   125741 non-null  object
 2   cum_runs                 125741 non-null  object
 3   cum_wickets              125741 non-null  object
 4   current_run_rate         125741 non-null  object
 5   required_run_rate        125741 non-null  object
 6   target                   125741 non-null  object
 7   batting_team_encoded     125741 non-null  object
 8   bowling_team_encoded     125741 non-null  object
 9   winner_encoded           125741 non-null  object
 10  venue_canonical_encoded  125741 non-null  object
 11  win                      125741 non-null  object
dtypes: object(12)
memory usage: 12.5+ MB


In [127]:
final_data.to_csv("final.csv", index=False) 

In [129]:
final_data.head()

,match_id,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team_encoded,bowling_team_encoded,winner_encoded,venue_canonical_encoded,win
124,335982,2,1,0,0.0,11.1,223,11,4,4,16,0
125,335982,2,2,0,12.0,11.142857142857144,223,11,4,4,16,0
126,335982,2,2,0,6.0,11.23728813559322,223,11,4,4,16,0
127,335982,2,3,0,6.0,11.282051282051283,223,11,4,4,16,0
128,335982,2,4,0,6.0,11.327586206896552,223,11,4,4,16,0
